In [21]:
# --- import libraries ---
import pandas as pd
import numpy as np
import os
import shutil
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError
from sklearn.preprocessing import StandardScaler
from lifelines import WeibullAFTFitter
from typing import Tuple, Optional
import matplotlib.pyplot as plt
import seaborn as sns
import sys

sys.path.append('../utilities')

PARTICIPANT_DATA_PATH = '../participant_data/'

## Method 1: Default Method

In [39]:
from preprocess import preprocess

index_event = "Borrow"
outcome_event = "Liquidated"
dataset_path = os.path.join(index_event, outcome_event)

train_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'train.csv'))
test_features_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'test_features.csv'))

X_train, y_train, X_test_processed = preprocess(train_df, test_features_df)

lifelines_train_df = pd.concat([X_train, y_train.reset_index(drop=True)], axis=1)
lifelines_train_df = lifelines_train_df.loc[lifelines_train_df['timeDiff'] > 0].copy()

In [40]:
train_df.head()

,userBorrowSum,marketWithdrawAvgAmount,marketDepositAvgAmountUSD,sinDayOfMonth,userSecondsSinceFirstTransaction,timeOfDay,userActiveDaysWeekly,userLiquidationCount,userActiveDaysMonthly,userRepayCount,...,userBorrowAvgAmountUSD,userReserveMode,userSecondsSincePreviousTransaction,marketBorrowSumUSD,userRepaySum,logAmountUSD,marketDepositCount,marketRepayAvgAmountUSD,sinQuarter,marketLiquidationAvgAmount
0,1.00000,0.000,1.160624,0.207912,147302.0,8.669167,3.0,0,3.0,0,...,0.999892,DAI,64356.0,0.999892,0.0,0.693093,5.0,0.0,1.0,0.0
1,0.20000,1.999,1.199348,-0.207912,3185.0,8.832500,1.0,0,1.0,0,...,0.199924,DAI,3185.0,1.199815,0.0,0.182258,9.0,0.0,1.0,0.0
2,1.00000,1.999,5.535831,-0.207912,270.0,10.563333,1.0,0,1.0,0,...,1.000053,DAI,270.0,2.199868,0.0,0.693174,11.0,0.0,1.0,0.0
3,6.00000,1.999,5.535831,-0.207912,310.0,10.574444,1.0,0,1.0,0,...,3.000026,USDC,40.0,7.199868,0.0,1.791759,11.0,0.0,1.0,0.0
4,0.79299,1.999,7.933243,-0.207912,328.0,11.998333,1.0,0,1.0,0,...,0.793127,DAI,328.0,7.992995,0.0,0.583961,13.0,0.0,1.0,0.0


In [24]:
# --- import parametric covariates (multivariate) survival models from lifelines ---
from lifelines import WeibullAFTFitter

model = WeibullAFTFitter(penalizer=0.1)
model.fit(lifelines_train_df, duration_col='timeDiff', event_col='status')

# model.print_summary(3)

<lifelines.WeibullAFTFitter: fitted with 885908 total observations, 861864 right-censored observations>

In [25]:
from lifelines.utils import concordance_index

# y_test has columns: timeDiff (duration) and status (event indicator)
predictions = model.predict_median(X_train)

c_index = concordance_index(
    event_times = lifelines_train_df["timeDiff"],   
    predicted_scores = predictions,                
    event_observed = lifelines_train_df["status"]   
)

print("c-index:", c_index)

c-index: 0.7971472134567833


In [26]:
summary_df = model.summary
summary_df.head()

coef  exp(coef)  se(coef)  coef lower 95%  \
param   covariate                                                        
lambda_ amount           0.039217   1.039996  0.003270        0.032809   
        amountUSD        0.046413   1.047507  0.003281        0.039982   
        cosDayOfMonth   -0.000432   0.999568  0.003066       -0.006442   
        cosDayOfQuarter -0.008813   0.991226  0.003080       -0.014850   
        cosDayOfWeek    -0.004510   0.995500  0.003065       -0.010518   

                         coef upper 95%  exp(coef) lower 95%  \
param   covariate                                              
lambda_ amount                 0.045625             1.033353   
        amountUSD              0.052844             1.040792   
        cosDayOfMonth          0.005578             0.993579   
        cosDayOfQuarter       -0.002776             0.985260   
        cosDayOfWeek           0.001497             0.989538   

                         exp(coef) upper 95%  cmp to          z             p  \
param   covariate                                                               
lambda_ amount                      1.046682     0.0  11.994526  3.795848e-33   
        amountUSD                   1.054265     0.0  14.144912  2.007665e-45   
        cosDayOfMonth               1.005594     0.0  -0.140779  8.880448e-01   
        cosDayOfQuarter             0.997228     0.0  -2.861151  4.221054e-03   
        cosDayOfWeek                1.001498     0.0  -1.471630  1.411209e-01   

                           -log2(p)  
param   covariate                    
lambda_ amount           107.699205  
        amountUSD        148.481245  
        cosDayOfMonth      0.171296  
        cosDayOfQuarter    7.888181  
        cosDayOfWeek       2.824996

## Method 2: Select Relevant Features & Train

In [27]:
covariates = summary_df.index.tolist()      
hr = summary_df['exp(coef)'].to_list() # hazard ratio
p_values = summary_df['p'].to_list() # p values
len(covariates), len(hr), len(p_values)                           

(80, 80, 80)

In [28]:
##-- select features with both low p-value and meaningful HR --##
    # keep covariates with HR < 0.8 or HR > 1.2, provided p < 0.05.
    # be cautious with HR ≈ 1.0 (like 0.98–1.02) -> effect is negligible

# coef -> regression coefficient (β)
# exp(coef) -> Hazard Ratio (HR) = exp(β) (row['p'] >= 0.001) and 

cols_to_keep = []
for index, row in summary_df.iterrows():
    if row['p'] <= 0.00001: # means only features that change hazard by less than ±20% are dropped
        cols_to_keep.append(index)

print(len(cols_to_keep))

40


In [29]:
col_names = [name for _, name in cols_to_keep]
len(col_names)

40

In [30]:
cols_to_keep.pop()
cols_to_keep.pop()

('lambda_', 'Intercept')

In [31]:
col_names = [name for _, name in cols_to_keep]
len(col_names)

38

In [41]:
index_event = "Borrow"
outcome_event = "Liquidated"
dataset_path = os.path.join(index_event, outcome_event)
new_train_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'train.csv'))

new_train_df = lifelines_train_df[col_names + ['timeDiff', 'status']]
new_train_df.shape

(885908, 40)

In [42]:
new_train_df.head()

,amount,amountUSD,dayOfYear,logAmountUSD,marketBorrowCount,marketDepositAvgAmount,marketDepositAvgAmountUSD,marketDepositCount,marketDepositSumUSD,marketWithdrawSumUSD,...,userRepayAvgAmount,userRepayAvgAmountUSD,userRepayCount,userRepaySum,userRepaySumUSD,userSecondsSinceFirstTransaction,userWithdrawAvgAmount,userWithdrawAvgAmountUSD,timeDiff,status
0,-0.357465,-0.420648,-1.161545,-1.254174,-2.240522,-2.163606,-2.099994,-1.577749,-1.423436,-1.34988,...,-0.46772,-0.486161,-0.402879,-0.398471,-0.397683,-0.785666,-0.244026,-0.295377,75264503.0,0.0
1,-0.357486,-0.420668,-1.142780,-1.382937,-2.240461,-2.163563,-2.099970,-1.577674,-1.423436,-1.34988,...,-0.46772,-0.486161,-0.402879,-0.398471,-0.397683,-0.797257,-0.244026,-0.295377,75091115.0,0.0
2,-0.357465,-0.420648,-1.142780,-1.254154,-2.240399,-2.159853,-2.097217,-1.577637,-1.423435,-1.34988,...,-0.46772,-0.486161,-0.402879,-0.398471,-0.397683,-0.797492,-0.244026,-0.295377,75084884.0,0.0
3,-0.357360,-0.420546,-1.142780,-0.977241,-2.240338,-2.159853,-2.097217,-1.577637,-1.423435,-1.34988,...,-0.46772,-0.486161,-0.402879,-0.398471,-0.397683,-0.797489,-0.244026,-0.295377,75084844.0,0.0
4,-0.357470,-0.420653,-1.142780,-1.281682,-2.240277,-2.158599,-2.095696,-1.577600,-1.423435,-1.34988,...,-0.46772,-0.486161,-0.402879,-0.398471,-0.397683,-0.797487,-0.244026,-0.295377,75079718.0,0.0


In [34]:
# from preprocess import preprocess

# print("Preprocessing data...")
# X_train_new, y_train_new, X_test_processed = preprocess(new_train_df, test_features_df)
# new_lifelines_train_df = pd.concat([X_train_new, y_train_new.reset_index(drop=True)], axis=1)
# new_lifelines_train_df.head()

In [43]:
# --- import parametric covariates (multivariate) survival models from lifelines ---
from lifelines import WeibullAFTFitter

model = WeibullAFTFitter(penalizer=0.1)
print("Training model on full training data...")
model.fit(new_train_df, duration_col='timeDiff', event_col='status')

# model.print_summary(3)

Training model on full training data...


<lifelines.WeibullAFTFitter: fitted with 885908 total observations, 861864 right-censored observations>

In [44]:
from lifelines.utils import concordance_index

predictions = model.predict_median(X_train)

c_index = concordance_index(
    event_times = new_train_df["timeDiff"],   
    predicted_scores = predictions,                # negate: lower predicted survival → higher risk
    event_observed = new_train_df["status"]   
)

print("C-index:", c_index)

C-index: 0.7986676231435561


In [37]:
# model.print_summary()